### AI Financial Analyst

#### Member 2

### Notebook 5 - Embeddings and FAISS Vector Database

---

#### Objective

The objective of this notebook is to generate semantic embeddings for the
processed SEC narrative chunks and construct a FAISS vector database to
support semantic retrieval in the Retrieval-Augmented Generation (RAG)
pipeline.

The workflow includes:

- Loading processed narrative chunks
- Generating sentence embeddings
- Building a FAISS vector index
- Validating semantic retrieval
- Saving the embeddings and vector database

---

#### CRISP-DM Phase

**Modelling**

#### Step 1 - Installing Required Libraries

In [1]:
# Install Required Libraries
!pip install sentence-transformers faiss-cpu

#### Step 2 - Importing Libraries

In [2]:
# Importing Libraries

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

print("Libraries imported successfully.")

Libraries imported successfully.


#### Step 3 - Loading Processed Chunks

In [3]:
# Loading Processed Chunks

chunks_df = pd.read_parquet(
    "../data/processed/processed_chunks.parquet"
)

print("Processed chunks loaded successfully.")
display(chunks_df.head())

Processed chunks loaded successfully.


,ticker,company_name,cik,year,section,chunk_id,chunk_text,word_count
0,APTV,Aptiv PLC,1521332,2014,section_1,0,"ITEM 1. BUSINESS “Delphi,” the “Company,” “we,...",300
1,APTV,Aptiv PLC,1521332,2014,section_1,1,"focus on these markets, particularly China, wh...",300
2,APTV,Aptiv PLC,1521332,2014,section_1,2,"PBGC were redeemed, respectively, for approxim...",300
3,APTV,Aptiv PLC,1521332,2014,section_1,3,engine management systems including fuel handl...,300
4,APTV,Aptiv PLC,1521332,2014,section_1,4,and products. Our customer base includes all 2...,300


#### Step 4 - Validate Dataset

In [4]:
# Validating Dataset

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print("Chunks :", len(chunks_df))
print("Companies :", chunks_df["ticker"].nunique())
print("Years :", sorted(chunks_df["year"].unique()))
print("Sections :", chunks_df["section"].unique())

DATASET SUMMARY
Chunks : 10009
Companies : 22
Years : ['2014', '2015', '2016', '2017', '2018']
Sections : ['section_1' 'section_1A' 'section_7']


#### Step 4A - Validate Narrative Chunks

Before generating embeddings, the chunk dataset is checked for missing or
empty narrative chunks.

This validation ensures that embeddings are generated only from meaningful
text.

In [5]:
# Validate Narrative Chunks

print("=" * 60)
print("TEXT QUALITY")
print("=" * 60)

missing = (
    chunks_df["chunk_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Empty Chunks : {missing}")

TEXT QUALITY
Empty Chunks : 0


#### Step 5 - Loading Embedding Model

A pre-trained Sentence Transformer model is used to generate dense vector
representations of the narrative chunks.

For this project, the **all-MiniLM-L6-v2** model is selected because it
provides a good balance between computational efficiency and semantic
representation quality.

In [6]:
# Loading Sentence Transformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [7]:
print("=" * 60)
print("EMBEDDING MODEL")
print("=" * 60)

print("Model Name :", "all-MiniLM-L6-v2")

EMBEDDING MODEL
Model Name : all-MiniLM-L6-v2


#### Step 6 - Generating Embeddings

In [8]:
# Generating Embeddings

embeddings = model.encode(
    chunks_df["chunk_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

print("Embeddings generated successfully.")

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Embeddings generated successfully.


#### Step 7 - Validating Embeddings

In [9]:
# Validating Embeddings

print("=" * 60)
print("EMBEDDING SUMMARY")
print("=" * 60)

print("Embedding Shape :", embeddings.shape)
print("Embedding Dimension :", embeddings.shape[1])
print("Embedding Data Type :", embeddings.dtype)


EMBEDDING SUMMARY
Embedding Shape : (10009, 384)
Embedding Dimension : 384
Embedding Data Type : float32


#### STEP 8 - Building FAISS Index

In [10]:
# Building FAISS Index

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)
print("FAISS index created successfully.")

FAISS index created successfully.


####  Step 9 - Validate FAISS Index

#### Step 10 - Testing Semantic Retrieval

In [12]:
# Testing Semantic Retrieval

query = "What are the major financial risks facing the company?"
query_embedding = model.encode([query])

distances, indices = index.search(
    query_embedding,
    k=3
)

results = chunks_df.iloc[indices[0]].copy()

results["distance"] = distances[0]

display(
    results[
        [
            "ticker",
            "year",
            "section",
            "chunk_text"
        ]
    ]
)

,ticker,year,section,chunk_text
5183,NOVT,2016,section_1A,Item 1A. Risk Factors The following risk facto...
3101,CLDX,2018,section_1,we are subject to a number of risks that you s...
4985,MAYS,2018,section_1A,The Company mitigates risks of tenants with le...


#### Step 10A - Embedding Summary

A summary of the generated embeddings and FAISS vector database is
displayed before the outputs are saved.

This provides a concise overview of the modelling stage.

In [13]:
# Embedding Summary

summary = pd.DataFrame({

    "Metric":[
        "Narrative Chunks",
        "Embedding Dimension",
        "Vectors Stored"
    ],

    "Value":[
        len(chunks_df),
        embeddings.shape[1],
        index.ntotal
    ]

})

display(summary)

,Metric,Value
0,Narrative Chunks,10009
1,Embedding Dimension,384
2,Vectors Stored,10009


#### Step 11 - Saving Embeddings

In [14]:
# Saving Embeddings

np.save(
    "../data/processed/chunk_embeddings.npy",
    embeddings
)

print("Embeddings saved successfully.")

Embeddings saved successfully.


#### Step 12 - Saving FAISS Index

In [15]:
# Saving FAISS Index

faiss.write_index(

    index,
    "../data/processed/faiss_index.bin"

)
print("FAISS index saved successfully.")

FAISS index saved successfully.


#### Step 13 - Saving Chunk Metadata

In [16]:
# Saving Chunk Metadata

chunks_df.to_parquet(
    "../data/processed/chunk_metadata.parquet",
    index=False
)

print("Chunk metadata saved successfully.")

Chunk metadata saved successfully.


#### Conclusion

This notebook successfully generated semantic vector representations for
the processed SEC narrative chunks and constructed a FAISS vector database
to support semantic retrieval.

The completed workflow included:

- Loading processed narrative chunks
- Generating sentence embeddings using the all-MiniLM-L6-v2 model
- Building and validating a FAISS vector index
- Performing semantic retrieval using a sample query
- Saving embeddings, metadata and the vector database

The generated vector database forms the retrieval component of the
Retrieval-Augmented Generation (RAG) architecture and will be integrated
with the Large Language Model in the next notebook.